<a href="https://colab.research.google.com/github/Quantium/Zeus-Notebooks/blob/main/ZeusPipeline.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
#@title Installations
# !pip install rich #Already in colab vms
!pip install rich-tools pandas


In [ ]:
#@title Base Libraries
import os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import joblib

In [ ]:
#@title Select for Dark Mode
import seaborn as sns

darkmode_selector = True #@param {type:"boolean"}
if darkmode_selector:
  plt.style.use('seaborn-v0_8-whitegrid')

  plt.style.use('dark_background')
  plt.rcParams.update({
      'axes.edgecolor': 'white',
      'axes.labelcolor': 'white',
      'text.color': 'white',
      'xtick.color': 'white',
      'ytick.color': 'white',
      'grid.color': 'lightgray',
      'grid.linestyle': '--',
      'grid.linewidth': 0.5,
      'lines.linewidth': 1.5,
      'lines.markersize': 5,
      'font.size': 10,
      'figure.figsize': (8, 6),
      'savefig.dpi': 300,
      'savefig.bbox': 'tight',
      'legend.edgecolor': 'white',
      'legend.frameon': True,
      'legend.fancybox': False,
  })

  #Para as matrices de correlación
  dark_cmap = sns.color_palette("binary", as_cmap=True)

In [ ]:
#@title ML Libraries

from sklearn.model_selection import train_test_split, StratifiedKFold, KFold, cross_validate, GridSearchCV,RandomizedSearchCV
from sklearn.feature_selection import SelectKBest, chi2, VarianceThreshold
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import mean_absolute_error, mean_squared_error,r2_score, RocCurveDisplay,accuracy_score, roc_curve, auc, classification_report,confusion_matrix,precision_recall_curve, precision_recall_fscore_support,fbeta_score, ConfusionMatrixDisplay
from sklearn.pipeline import Pipeline
from sklearn.linear_model import Lasso, LinearRegression, HuberRegressor

from scipy.stats import kurtosis, skew

In [ ]:
#@title Función para guardar objetos con joblib
## Crea o carga un objeto, ideal para objetos que toman mucho tiempo en generarse
def create_or_load(folder_path, filename, creation_function):
  file_path=os.path.join(folder_path, filename)
  if not os.path.exists(file_path):
      print(f"Joblib file not found. Creating {file_path}")
      data = creation_function()
      joblib.dump(data, file_path)

  print(f"Joblib file found.{file_path}")
  return joblib.load(file_path)

In [ ]:
#@title Google Drive
from google.colab import drive
from google.colab import auth
import gspread
from google.auth import default
import pandas as pd

auth.authenticate_user()
creds, _ = default()
gc = gspread.authorize(creds)

drive.mount('/content/drive')
working_folder = "/content/drive/MyDrive/Darovi Consulting/Zeus/reextraccindedatosproyectohades" # @param {type:"string"}

working_file = 'Columnas para Recibo OC  lineas y pedido'

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
#@title Líneas DataFrame
lineas = gc.open(working_file).worksheet('lineas')

rows = lineas.get_all_values()
lineasSucios_df = pd.DataFrame.from_records(rows[1:], columns=rows[0])
lineas_df=lineasSucios_df[['DATE','ID','CODIGO_EMPRESA','ORGANIZACION_COMPRAS','SERIE_NUMERACION','NUMERO_PEDIDO','CODIGO_ARTICULO','DESCRIPCION','FECHA_ENTREGA','FECHA_ENTREGA_CONFIRM','UNIDADES_PEDIDAS','STATUS_CIERRE','STATUS_APROBADO','FORMA_FARMACEUTICA','D_CODIGO_FAMILIA']]
lineas_df

,DATE,ID,CODIGO_EMPRESA,ORGANIZACION_COMPRAS,SERIE_NUMERACION,NUMERO_PEDIDO,CODIGO_ARTICULO,DESCRIPCION,FECHA_ENTREGA,FECHA_ENTREGA_CONFIRM,UNIDADES_PEDIDAS,STATUS_CIERRE,STATUS_APROBADO,FORMA_FARMACEUTICA,D_CODIGO_FAMILIA
0,,,1,5450,PCM,29901,010.000.0566.00,METILDOPA ( L-ALFAMETILDOPA) TABLETAS 250 MG E...,05/05/2026,,460,E,S,,MEDICAMENTOS
1,,,1,5450,PCM,29863,010.000.0569.00,NITROPRUSIATO DE SODIO SOLUCIÓN INYEC. 50 MG E...,04/05/2026,,5,E,S,,MEDICAMENTOS
2,,,1,5450,PCM,29864,010.000.0569.00,NITROPRUSIATO DE SODIO SOLUCIÓN INYEC. 50 MG E...,04/05/2026,,3,E,S,,MEDICAMENTOS
3,,,1,5450,PCM,29870,010.000.0569.00,NITROPRUSIATO DE SODIO SOLUCIÓN INYEC. 50 MG E...,04/05/2026,,3,E,S,,MEDICAMENTOS
4,,,1,5450,PCM,29829,010.000.2308.00,FUROSEMIDA. SOLUCIÓN INYECTABLE. CADA AMPOLLET...,04/05/2026,,5018,E,S,,MEDICAMENTOS
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
8283,,,1,5450,PCM,29854,010.000.6231.00,PERINDOPRIL/AMLODIPINO. COMPRIMIDOS CADA COMPR...,04/05/2026,,10,E,S,,MEDICAMENTOS
8284,,,1,5450,PCM,29869,060.182.0160,CEMENTOS. IONÓMERO DE VIDRIO I. PARA CEMENTACI...,04/05/2026,,27,E,S,,MATERIAL DE CURACIÓN
8285,,,1,5460,OP6,1706,25411UG01438,"CATETER VENOSO CENTRAL PICC, DE POLIURETANO, 5...",04/05/2026,,6,Z,S,,MATERIAL DE CURACIÓN
8286,,,1,5460,OP6,1704,25411UG00795,TERMOMETRO DIGITAL PUNTA RIGIDA SUAVE AL TACTO...,04/05/2026,,1080,Z,S,,MATERIAL DE CURACIÓN


In [ ]:
#@title Pedidos DataFrame
pedidos = gc.open(working_file).worksheet('pedidos')

rows = pedidos.get_all_values()
pedidosSucio_df = pd.DataFrame.from_records(rows[1:], columns=rows[0])
pedidos_df=pedidosSucio_df[['DATE','ID','CODIGO_EMPRESA','ORGANIZACION_COMPRAS','TIPO_PEDIDO','SERIE_NUMERACION','NUMERO_PEDIDO','CODIGO_PROVEEDOR','FECHA_PEDIDO','CODIGO_ALMACEN','USUARIO_ALTA','FECHA_ALTA','STATUS_CIERRE','STATUS_APROBADO']]
pedidos_df

,DATE,ID,CODIGO_EMPRESA,ORGANIZACION_COMPRAS,TIPO_PEDIDO,SERIE_NUMERACION,NUMERO_PEDIDO,CODIGO_PROVEEDOR,FECHA_PEDIDO,CODIGO_ALMACEN,USUARIO_ALTA,FECHA_ALTA,STATUS_CIERRE,STATUS_APROBADO
0,,,1,7151,M,D51,2093,804210,05/05/2026,ME004,PEVAJE,05/05/2026,E,N
1,,,1,7151,M,D51,2094,804210,05/05/2026,ME004,PEVAJE,05/05/2026,E,N
2,,,1,7151,M,D51,2095,804210,05/05/2026,ME004,PEVAJE,05/05/2026,E,N
3,,,1,7151,M,D51,2096,804210,05/05/2026,ME004,PEVAJE,05/05/2026,E,N
4,,,1,7151,M,D51,2097,804210,05/05/2026,ME004,PEVAJE,05/05/2026,E,N
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
170,,,1,5460,M,OP6,1782,250,06/05/2026,5460D,GOMFLOJED,05/05/2026,E,S
171,,,1,5460,M,OP6,1783,250,06/05/2026,5460D,GOMFLOJED,05/05/2026,E,S
172,,,1,5460,M,OP6,1784,250,06/05/2026,5460D,GOMFLOJED,05/05/2026,E,S
173,,,1,5460,M,OP6,1785,250,06/05/2026,5460D,GOMFLOJED,05/05/2026,E,S


In [ ]:
#@title Calendario de citas
 DataFrame

In [ ]:
#@title Surtido DataFrame

In [ ]:
#@title Embarques DataFrame